In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# IBM Circuit function

*Lihat [referensi API](https://docs.quantum.ibm.com/api/functions/ibm-circuit-function)*

> **Note:** * Qiskit Functions adalah fitur eksperimental yang hanya tersedia untuk pengguna IBM Quantum&reg; Premium Plan, Flex Plan, dan On-Prem (via IBM Quantum Platform API) Plan. Fitur ini masih dalam status preview release dan dapat berubah sewaktu-waktu.

## Ikhtisar
IBM&reg; Circuit function menerima [abstract PUBs](/guides/primitive-input-output#pubs) sebagai input, dan mengembalikan nilai ekspektasi yang telah dimitigasi sebagai output. Circuit function ini mencakup pipeline otomatis dan tersesuaikan untuk membantu para peneliti fokus pada penemuan algoritma dan aplikasi.

## Deskripsi
Setelah Anda mengirimkan PUB, sirkuit abstrak dan observable Anda akan ditranspilasi secara otomatis, dieksekusi pada hardware, dan diproses pasca-eksekusi untuk mengembalikan nilai ekspektasi yang telah dimitigasi. Untuk melakukannya, ini menggabungkan alat-alat berikut:

- [Qiskit Transpiler Service](/guides/qiskit-transpiler-service), termasuk pemilihan otomatis transpilasi berbasis AI dan heuristik untuk menerjemahkan sirkuit abstrakmu ke sirkuit ISA yang dioptimalkan untuk hardware
- [Suppression dan mitigasi error yang diperlukan untuk komputasi skala utilitas](/guides/error-mitigation-and-suppression-techniques), termasuk measurement dan gate twirling, dynamical decoupling, Twirled Readout Error eXtinction (TREX), Zero-Noise Extrapolation (ZNE), dan Probabilistic Error Amplification (PEA)
- [Qiskit Runtime Estimator](/guides/get-started-with-estimator), untuk mengeksekusi ISA PUBs pada hardware dan mengembalikan nilai ekspektasi yang telah dimitigasi

![IBM Circuit function](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)

## Mulai
Autentikasi menggunakan [API key](http://quantum.cloud.ibm.com/) Anda dan pilih Qiskit Function sebagai berikut. (Snippet ini mengasumsikan Anda sudah [menyimpan akun Anda](/guides/functions#install-qiskit-functions-catalog-client) ke lingkungan lokal Anda.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## Contoh
Untuk memulai, coba contoh dasar ini:

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

Periksa [status](/guides/functions#check-job-status) workload Qiskit Function-mu atau ambil [hasil](/guides/functions#retrieve-results) sebagai berikut:

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/estimator-input-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


Hasilnya memiliki format yang sama dengan [hasil Estimator](/guides/estimator-input-output):

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

### Contoh mitigation level
Contoh berikut mendemonstrasikan pengaturan mitigation level:

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


Dalam contoh berikut, mengatur mitigation level ke 1 awalnya mematikan mitigasi ZNE, tetapi mengatur `zne_mitigation` ke `True` menimpa pengaturan relevan dari `mitigation_level`.

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

### Contoh output
Snippet kode berikut mendeskripsikan format `PrimitiveResult` (dan `PubResult` terkait).